# Assigment 1 - Part 2

Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import STL10
import torchvision.transforms as tt
from torchvision.models import resnet18, mobilenet_v3_large
from torchvision.models import ResNet18_Weights, MobileNet_V3_Large_Weights
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

torch.manual_seed(42)
np.random.seed(42)

STL-10 dataset download and preparation

In [ ]:
# Resize to 224 to match ImageNet-pretrained input size
transform = tt.Compose([
    tt.Resize(224),
    tt.ToTensor(),
    tt.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

training_data = STL10(root='./data', split='train', download=True, transform=transform)
testing_data = STL10(root='./data', split='test', download=True, transform=transform)

# Train/validation split (80/20)
train_size = int(len(training_data) * 0.8)
val_size = len(training_data) - train_size
train_set, val_set = torch.utils.data.random_split(
    training_data, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

batch_size = 64
train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(testing_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
categories = training_data.classes
print(f'Device: {device}')
print(f'Train samples: {len(train_set)}, Val samples: {len(val_set)}, Test samples: {len(testing_data)}')

100%|██████████| 2.64G/2.64G [02:48<00:00, 15.7MB/s]


Device: cuda
Train samples: 4000, Val samples: 1000, Test samples: 8000


Training and Evaluation functions

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    train_losses, val_losses, val_accuracies = [], [], []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        epoch_train_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)

        model.eval()
        running_val_loss = 0.0
        correct = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()

        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        epoch_val_acc = 100.0 * correct / len(val_loader.dataset)
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_acc)

        print(f'Epoch [{epoch+1}/{num_epochs}] Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%')

    return train_losses, val_losses, val_accuracies

def evaluate_model(model, test_loader, categories, model_name='Model'):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)
    class_acc = cm.diagonal() / cm.sum(axis=1)
    mca = 100.0 * np.mean(class_acc)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')

    print(f'=== {model_name} Test Results ===')
    print(f'mean-Class Accuracy (mCA): {mca:.2f}%')
    print(f'Macro Precision: {precision*100:.2f}%')
    print(f'Macro Recall: {recall*100:.2f}%')
    print(f'Macro F1-Score: {f1*100:.2f}%')

    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=categories, yticklabels=categories)
    plt.title(f'{model_name} - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

    return {'mCA': mca, 'Precision': precision * 100, 'Recall': recall * 100, 'F1': f1 * 100}

def plot_training_history(train_losses, val_losses, val_accs, title='Model'):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(val_accs, label='Val Accuracy', color='green')
    plt.title(f'{title} - Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.tight_layout()
    plt.show()

Hyperparameters

In [ ]:
lr = 1e-4
num_epochs = 20
criterion = nn.CrossEntropyLoss()

ResNet-18 training from scratch

In [ ]:
resnet_scratch = resnet18(weights=None, num_classes=10).to(device)
optimizer_resnet_scratch = optim.Adam(resnet_scratch.parameters(), lr=lr, weight_decay=1e-4)

t_loss, v_loss, v_acc = train_model(resnet_scratch, train_loader, val_loader, criterion, optimizer_resnet_scratch, num_epochs=num_epochs)
plot_training_history(t_loss, v_loss, v_acc, title='ResNet-18 (Scratch)')
metrics_resnet_scratch = evaluate_model(resnet_scratch, test_loader, categories, model_name='ResNet-18 (Scratch)')

MobileNetV3 Large training from scratch

In [ ]:
mobilenet_scratch = mobilenet_v3_large(weights=None, num_classes=10).to(device)
optimizer_mobilenet_scratch = optim.Adam(mobilenet_scratch.parameters(), lr=lr, weight_decay=1e-4)

t_loss, v_loss, v_acc = train_model(mobilenet_scratch, train_loader, val_loader, criterion, optimizer_mobilenet_scratch, num_epochs=num_epochs)
plot_training_history(t_loss, v_loss, v_acc, title='MobileNetV3 Large (Scratch)')
metrics_mobilenet_scratch = evaluate_model(mobilenet_scratch, test_loader, categories, model_name='MobileNetV3 Large (Scratch)')

ResNet-18 training with pre-trained weights

In [ ]:
resnet_pretrained = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
resnet_pretrained.fc = nn.Linear(resnet_pretrained.fc.in_features, 10)
resnet_pretrained = resnet_pretrained.to(device)

optimizer_resnet_pt = optim.Adam(resnet_pretrained.parameters(), lr=lr, weight_decay=1e-4)
t_loss, v_loss, v_acc = train_model(resnet_pretrained, train_loader, val_loader, criterion, optimizer_resnet_pt, num_epochs=num_epochs)
plot_training_history(t_loss, v_loss, v_acc, title='ResNet-18 (Pretrained)')
metrics_resnet_pt = evaluate_model(resnet_pretrained, test_loader, categories, model_name='ResNet-18 (Pretrained)')

MobileNetV3 Large training with pre-trained weights

In [ ]:
mobilenet_pretrained = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1)
mobilenet_pretrained.classifier[3] = nn.Linear(mobilenet_pretrained.classifier[3].in_features, 10)
mobilenet_pretrained = mobilenet_pretrained.to(device)

optimizer_mobilenet_pt = optim.Adam(mobilenet_pretrained.parameters(), lr=lr, weight_decay=1e-4)
t_loss, v_loss, v_acc = train_model(mobilenet_pretrained, train_loader, val_loader, criterion, optimizer_mobilenet_pt, num_epochs=num_epochs)
plot_training_history(t_loss, v_loss, v_acc, title='MobileNetV3 Large (Pretrained)')
metrics_mobilenet_pt = evaluate_model(mobilenet_pretrained, test_loader, categories, model_name='MobileNetV3 Large (Pretrained)')

# Middle Fusion

In [ ]:
class MiddleFusionNet(nn.Module):
    def __init__(self, resnet, mobilenet, num_classes=10):
        super().__init__()
        # Feature extractors (without final pooling and classifier)
        self.resnet_features = nn.Sequential(*list(resnet.children())[:-2])
        self.mobilenet_features = mobilenet.features

        # Freeze extractor weights
        for p in self.resnet_features.parameters():
            p.requires_grad = False
        for p in self.mobilenet_features.parameters():
            p.requires_grad = False

        # Concatenation fusion: 512 (ResNet) + 960 (MobileNetV3) = 1472 channels
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(1472, num_classes)
        )

    def forward(self, x):
        f1 = self.resnet_features(x)
        f2 = self.mobilenet_features(x)
        # Align spatial dims if needed
        if f1.shape[2:] != f2.shape[2:]:
            f1 = nn.functional.adaptive_avg_pool2d(f1, f2.shape[2:])
        f = torch.cat((f1, f2), dim=1)
        return self.classifier(f)

fusion_model = MiddleFusionNet(resnet_pretrained, mobilenet_pretrained, num_classes=10).to(device)
optimizer_fusion = optim.Adam(fusion_model.classifier.parameters(), lr=lr, weight_decay=1e-4)

t_loss, v_loss, v_acc = train_model(fusion_model, train_loader, val_loader, criterion, optimizer_fusion, num_epochs=num_epochs)
plot_training_history(t_loss, v_loss, v_acc, title='Middle Fusion')
metrics_fusion = evaluate_model(fusion_model, test_loader, categories, model_name='Middle Fusion')

Metrics comparison

In [ ]:
results = {
    'ResNet-18 (Scratch)': metrics_resnet_scratch,
    'MobileNetV3 (Scratch)': metrics_mobilenet_scratch,
    'ResNet-18 (Pretrained)': metrics_resnet_pt,
    'MobileNetV3 (Pretrained)': metrics_mobilenet_pt,
    'Middle Fusion': metrics_fusion
}
df_results = pd.DataFrame(results).T
print(df_results.round(2))

                            mCA  Precision  Recall     F1
ResNet-18 (Scratch)       59.34      63.09   59.34  58.94
MobileNetV3 (Scratch)     53.08      57.66   53.08  52.79
ResNet-18 (Pretrained)    94.99      94.98   94.99  94.98
MobileNetV3 (Pretrained)  95.36      95.39   95.36  95.35
Middle Fusion             96.14      96.14   96.14  96.13
